# small\_fable — hard-reasoning traces → SFT → GRPO on Colab, with resume

Trains the joint planner+executor on your **hard reasoning traces** (`traces/`), where the plan is
load-bearing by design and the executor learns to **reason in prose, then commit a `FINAL ANSWER`**.

- **Stage 0** (fast, CPU): adapt traces + answer-keys → joint-SFT corpus + planner vocab.
- **Stage 1** SFT · **2a** rollouts · **2b** off-policy GRPO — all checkpoint to HF every ~10 min
  and `--resume`, so a Colab crash just needs a re-run.
- **Eval**: in-distribution (held-out set 1000) **and generalization** on sets 2000 (one-step-deeper)
  and 3000 (flipped-answer) — your memorize-vs-reason test.

> **Before running:** T4 GPU + a Hugging Face **write** token as a Colab secret `HF_TOKEN`. Then
> `Runtime → Run all`. If Colab dies, just Run all again — finished stages skip, the dead one resumes.


## 0 · GPU check

In [ ]:
!nvidia-smi || echo 'NO GPU — Runtime → Change runtime type → T4 GPU, then re-run.'


## 1 · Clone the repo & install deps

In [ ]:
import os
REPO = 'https://github.com/sinha-k-prat/small_fable.git'
if not os.path.isdir('small_fable'):
    !git clone -q $REPO
else:
    !cd small_fable && git pull -q
%cd small_fable
!pip install -q -r requirements.txt huggingface_hub
!pip uninstall -y torchao >/dev/null 2>&1; echo 'torchao removed (peft/torchao version-clash workaround)'
print('\nReady.')


## 2 · Hugging Face setup
Caches your `HF_TOKEN` so the `!python` trainers can push, and sets the CUDA allocator to reduce OOMs.

In [ ]:
import os
from huggingface_hub import HfApi, create_repo, snapshot_download, login
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from huggingface_hub import notebook_login; notebook_login()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False); os.environ['HF_TOKEN'] = HF_TOKEN
api = HfApi()
HF_REPO = f"{api.whoami()['name']}/small_fable-planner"
create_repo(HF_REPO, repo_type='model', exist_ok=True, private=False)
print('HF repo:', f'https://huggingface.co/{HF_REPO}')
print('>>> Use this HF_REPO in inference_colab.ipynb:', HF_REPO)

def pull_ckpt(sub):
    try:
        snapshot_download(repo_id=HF_REPO, allow_patterns=[f'{sub}/*'], local_dir='.')
        ok = os.path.exists(os.path.join(sub, 'train_state.pt'))
        print(f'[pull] {sub}: ' + ('resumable checkpoint found' if ok else 'no resumable state — fresh'))
    except Exception as e:
        print(f'[pull] {sub}: nothing to resume ({e})')

def push_file(path, repo_path=None):
    try:
        api.upload_file(path_or_fileobj=path, path_in_repo=repo_path or os.path.basename(path),
                        repo_id=HF_REPO, commit_message=f'upload {path}')
        print('[hf] pushed', path)
    except Exception as e:
        print('[hf] push skipped:', e)


## 2.5 · Stage 0 — build the SFT corpus from your traces (fast, CPU)
Joins `traces/*.jsonl` + `answers/*.jsonl` by instruction, makes the executor target the reasoning
prose **+ `FINAL ANSWER: <canonical>`**, and writes `plan_vocab.json` (your 23 refined primitives,
`FINALIZE` terminator). **Train on set 1000; hold sets 2000/3000 for the generalization test.**
No GPU, no base-model filtering — seconds, not hours.

In [ ]:
import json
TR = 'traces'; DATA = 'dataset/traces_sft.jsonl'
# planner vocab from ALL three sets (union) so the head covers every primitive the model may meet:
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_1000.jsonl $TR/hard_reasoning_traces_2000.jsonl $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_1000.jsonl $TR/answers_2000.jsonl $TR/answers_3000.jsonl --out /tmp/_all.jsonl --vocab_out plan_vocab.json
# training corpus = set 1000 only:
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_1000.jsonl --answers $TR/answers_1000.jsonl --out $DATA --vocab_out /tmp/_v.json
# held-out generalization eval sets:
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_2000.jsonl --answers $TR/answers_2000.jsonl --out dataset/traces_sft_2000.jsonl --vocab_out /tmp/_v.json
!python -u traces_to_sft.py --traces $TR/hard_reasoning_traces_3000.jsonl --answers $TR/answers_3000.jsonl --out dataset/traces_sft_3000.jsonl --vocab_out /tmp/_v.json
push_file('plan_vocab.json')   # inference needs it to size the planner head

N = sum(1 for _ in open(DATA))
HELD = 30; TRAIN = min(N - HELD, 500); ROLL_TRAIN = min(TRAIN, 120); MAX_RESP = 256
print('planner vocab:', json.load(open('plan_vocab.json'))['vocab'])
print(f'corpus rows={N}  ->  TRAIN={TRAIN}  HELD={HELD}  ROLL_TRAIN={ROLL_TRAIN}  MAX_RESP={MAX_RESP}')


## 3 · Stage 1 — Joint SFT  *(checkpoints + resumes via HF)*
Executor target is full reasoning prose (so `max_resp=256`, `bs=2` to fit a T4). Watch the held
`ablation_gap` each epoch — with load-bearing plans it should be **positive**. The final line reports
the real gap with a ✓/✗ verdict.

In [ ]:
pull_ckpt('joint_ckpt')
!python -u train_sft.py --data $DATA --train $TRAIN --held $HELD --epochs 4 --lr 2e-5 --bs 2 --max_resp $MAX_RESP --probe 12 --out joint_ckpt --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10


## 4 · Stage 2a — Offline rollouts

In [ ]:
pull_ckpt('joint_ckpt')
!python -u rollout_offline.py --sft_ckpt joint_ckpt --data $DATA --train $ROLL_TRAIN --group 8 --temp 1.2 --top_p 0.98 --max_resp $MAX_RESP --out rl_rollouts.jsonl --device cuda
push_file('rl_rollouts.jsonl')


## 5 · Stage 2b — Off-policy GRPO  *(checkpoints + resumes via HF)*
Watch `held_reward=` each inner epoch; final `|ΔL2| > 0` proves RL moved the adapter.

In [ ]:
pull_ckpt('joint_ckpt'); pull_ckpt('rl_ckpt')
import os; assert os.path.exists('rl_rollouts.jsonl'), 'run Stage 2a first (rollouts not on disk)'
!python -u train_grpo_offline.py --rollouts rl_rollouts.jsonl --sft_ckpt joint_ckpt --data $DATA --out rl_ckpt --inner_epochs 3 --lr 1e-4 --clip_eps 0.2 --beta_plan 1.0 --beta_ce 0.1 --held 16 --max_resp $MAX_RESP --device cuda --resume --hf_repo $HF_REPO --ckpt_every_min 10
print('All checkpoints on HF:', f'https://huggingface.co/{HF_REPO}/tree/main')


## 6 · Compare (in-distribution: held-out set 1000)
`--sample` is required to see RL effects.

In [ ]:
!python -u compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --data $DATA --train $TRAIN --held $HELD --max_resp $MAX_RESP --sample --device cuda


## 7 · Generalization — sets 2000 & 3000 (unseen variations)
The real test: the model trained on set 1000, evaluated on **one-step-deeper** (2000) and
**flipped-answer** (3000) variations. If accuracy holds (especially on 3000, where the answer flips),
the model is **reasoning**, not memorizing.

In [ ]:
print('================ SET 2000 (one-step-deeper) ================')
!python -u compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --data dataset/traces_sft_2000.jsonl --train 0 --held 100 --max_resp $MAX_RESP --sample --device cuda
print('\n================ SET 3000 (flipped-answer) ================')
!python -u compare.py --sft_ckpt joint_ckpt --rl_ckpt rl_ckpt --data dataset/traces_sft_3000.jsonl --train 0 --held 100 --max_resp $MAX_RESP --sample --device cuda


## 8 · Head-to-head on one hard prompt (SFT vs SFT+RL)

In [ ]:
import torch
from model_joint import JointModel, decode_plan
HARD_PROMPT = ('A snail is at the bottom of a 12-meter well. Each day it climbs 4 meters, '
               'but each night it slides back 3 meters. On which day does it first reach the top?')
def run(ckpt, label, n=2, temp=0.7):
    m = JointModel.from_checkpoint('Qwen/Qwen2.5-1.5B-Instruct', ckpt, device='cuda'); m.eval()
    print(f'\n{"="*70}\n{label}  ({ckpt})\n{"="*70}')
    with torch.no_grad():
        for k in range(n):
            p_ids, p_attn = m.batch_prompts([HARD_PROMPT])
            plan = m.sample_plan(p_ids, p_attn, temp=temp, sample=True)
            gen = m.generate_answer(p_ids, p_attn, plan, temp=temp, sample=True, max_new_tokens=256)
            print(f'  [sample {k+1}] plan: {" → ".join(decode_plan(plan[0]))}')
            print('   ', m.tok.decode(gen[0], skip_special_tokens=True).strip()[:600])
    del m; torch.cuda.empty_cache()
run('joint_ckpt', 'SFT only'); run('rl_ckpt', 'SFT + GRPO')
